In [ ]:
import pandas as pd

# URL of the raw CSV file on GitHub
# Note: Replacing 'github.com' with 'raw.githubusercontent.com' and removing '/blob' to get the raw file content
csv_url = 'https://raw.githubusercontent.com/TidyWeb/SkillMatch/data-snapshots/data/live_snapshots/20261021_2033_reprocessed/skills.csv'

try:
    skills  = pd.read_csv(csv_url)
    print("DataFrame loaded successfully!")
except Exception as e:
    print(f"Error loading CSV directly: {e}")
    print("Attempting to load via requests...")
    import requests
    import io
    response = requests.get(csv_url)
    skills = pd.read_csv(io.StringIO(response.text))

In [ ]:
# Display the first 5 rows and the dataframe's info
display(skills.head())
print("\nDataFrame Info:")
print("Shape:", skills.shape)
print("Columns:", skills.columns.tolist())
skills.info()
print(skills.isna().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set plotting style
sns.set_theme(style="whitegrid")

# 1. Calculate basic aggregates per vacancy_id
vacancy_skill_counts = skills.groupby('vacancy_id')['skill'].count().sort_values(ascending=False)

# 2. Prepare aggregated data for categorical columns 'strength' and 'confidence'
# Since these columns contain categorical text (e.g., 'essential', 'unclear', 'stated'),
# we aggregate their distributions per vacancy.
strength_by_vacancy = pd.crosstab(skills['vacancy_id'], skills['strength'])
confidence_by_vacancy = pd.crosstab(skills['vacancy_id'], skills['confidence'])

# Create a multi-plot figure
fig, axes = plt.subplots(3, 1, figsize=(12, 18), sharex=False)

# Plot 1: Number of skills per vacancy_id (Top 15)
vacancy_skill_counts.head(15).plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Top 15 Vacancies by Number of Required Skills', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Skills', fontsize=12)
axes[0].set_xlabel('Vacancy ID', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Strength distribution per vacancy_id (Top 15 vacancies)
top_vacancies = vacancy_skill_counts.head(15).index
strength_by_vacancy.loc[top_vacancies].plot(kind='bar', stacked=True, ax=axes[1], colormap='viridis')
axes[1].set_title('Strength Categories Distribution for Top 15 Vacancies', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count of Skills', fontsize=12)
axes[1].set_xlabel('Vacancy ID', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Strength')

# Plot 3: Confidence distribution per vacancy_id (Top 15 vacancies)
confidence_by_vacancy.loc[top_vacancies].plot(kind='bar', stacked=True, ax=axes[2], colormap='coolwarm')
axes[2].set_title('Confidence Categories Distribution for Top 15 Vacancies', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Count of Skills', fontsize=12)
axes[2].set_xlabel('Vacancy ID', fontsize=12)
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(title='Confidence')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Count occurrences of each unique skill
skill_counts = skills['skill'].value_counts()

# To make the pie chart readable, we will show the top 10 skills and group the rest as 'Others'
top_n = 10
top_skills = skill_counts.head(top_n)
others_count = skill_counts.iloc[top_n:].sum()

# Combine top skills with 'Others'
plot_data = top_skills.copy()
if others_count > 0:
    plot_data['Others'] = others_count

# Create the pie chart
plt.figure(figsize=(10, 10))
plt.pie(
    plot_data,
    labels=plot_data.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=plt.cm.tab20.colors[:len(plot_data)]
)

plt.title('Distribution of Specific Skills Across Vacancies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert GitHub URL to Raw URL
shown_url = 'https://raw.githubusercontent.com/TidyWeb/SkillMatch/data-snapshots/data/live_snapshots/20261021_2033_reprocessed/shown.csv'

# Load the shown.csv dataset
try:
    shown_df = pd.read_csv(shown_url)
    print("Loaded shown.csv successfully!")
except Exception as e:
    import requests
    import io
    response = requests.get(shown_url)
    shown_df = pd.read_csv(io.StringIO(response.text))
    print("Loaded shown.csv via requests fallback.")

# Set seaborn theme
sns.set_theme(style="whitegrid")

# Set up matplotlib subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 14))

# 1. Plot Employer Location counts
# Check the columns in the DataFrame to find exact matches for location and title
loc_col = [c for c in shown_df.columns if 'location' in c.lower() or 'employer' in c.lower()]
title_col = [c for c in shown_df.columns if 'title' in c.lower()]

# Use matched columns or fallback defaults
loc_column = loc_col[0] if loc_col else 'employer_text_location'
title_column = title_col[0] if title_col else 'title'

# Plot top employer locations
top_locations = shown_df[loc_column].value_counts().head(15)
sns.barplot(x=top_locations.values, y=top_locations.index, ax=axes[0], palette="viridis")
axes[0].set_title('Top 15 Employer Locations by Count of Vacancies', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count of Vacancies', fontsize=12)
axes[0].set_ylabel('Location', fontsize=12)

# Plot top job titles
top_titles = shown_df[title_column].value_counts().head(15)
sns.barplot(x=top_titles.values, y=top_titles.index, ax=axes[1], palette="magma")
axes[1].set_title('Top 15 Job Titles by Count of Vacancies', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count of Vacancies', fontsize=12)
axes[1].set_ylabel('Job Title', fontsize=12)

plt.tight_layout()
plt.show()